# ANALYSE-08 : Ramsey et van der Waerden — l'énumération qui prouve, et le mur où elle s'arrête

Carnet du pli 4 « Origami » (EPIC #19898, grain #19960). Deux théorèmes exacts de
petite combinatoire — $R(3,3) = 6$ et $W(2,3) = 9$ — y sont prouvés **par épuisement
brutal** : énumérer toutes les colorations, compter les survivantes, constater zéro.
Puis le même geste est tendu vers $R(4,4)$, où il meurt sur un mur de $2^{153}$
colorations — et l'argument probabiliste d'Erdős (1947) prend le relais : quand
compter devient impossible, **majorer** suffit pour l'existence.

Règle du carnet, héritée de la série : chaque affirmation numérique est **mesurée par
la cellule qui la précède**, ou citée avec sa source.

## 1. Ramsey $R(3,3)$ : l'énoncé, et la preuve par épuisement

**Énoncé.** Colorons chaque arête de $K_n$ (le graphe complet sur $n$ sommets) en
rouge ou bleu. $R(3,3)$ est le plus petit $n$ tel que **toute** coloration contienne un
triangle monochrome. La valeur $R(3,3) = 6$ se démontre en une soirée d'ordinateur :
énumérer les $2^{\binom{n}{2}}$ colorations et vérifier, pour chacune, ses
$\binom{n}{3}$ triangles.

La stratégie du carnet : monter $n = 4, 5, 6$ et regarder le décompte s'effondrer —
18 survivantes, puis 12, puis 0.

In [1]:
from itertools import combinations, product

def edges_of(n):
    # aretes (i < j) de K_n ; la position dans cette liste indexe la coloration
    return list(combinations(range(n), 2))

def mono_triangle(n, bits):
    # premier triangle monochrome de la coloration bits (None si aucune)
    E = edges_of(n)
    for tri in combinations(range(n), 3):
        cols = [bits[E.index((min(a, b), max(a, b)))] for a, b in combinations(tri, 2)]
        if cols[0] == cols[1] == cols[2]:
            return tri
    return None

def count_avoiders(n):
    # enumeration exhaustive : nombre de colorations sans triangle monochrome + temoin
    cnt, witness = 0, None
    for bits in product((0, 1), repeat=len(edges_of(n))):
        if mono_triangle(n, bits) is None:
            cnt += 1
            if witness is None:
                witness = ''.join(map(str, bits))
    return cnt, witness

n = 4
cnt, wit = count_avoiders(n)
ne = len(edges_of(n))
print(f"K_{n} : {ne} aretes, {2**ne} colorations")
print(f"sans triangle monochrome : {cnt} (exemple : {wit})")

K_4 : 6 aretes, 64 colorations
sans triangle monochrome : 18 (exemple : 001100)


### Lecture du résultat

Sur les $64$ colorations de $K_4$, dix-huit échappent au triangle monochrome — donc
$R(3,3) > 4$. Rien de surprenant : avec six arêtes, une coloration a beaucoup de
liberté. Ce premier palier fixe le geste : **le comptage exact est la preuve**, pas
une estimation. La question est de savoir jusqu'où il monte.

In [2]:
n = 5
cnt, wit = count_avoiders(n)
ne = len(edges_of(n))
print(f"K_{n} : {ne} aretes, {2**ne} colorations, sans triangle monochrome : {cnt}")

# Structure des survivantes : le rouge est-il toujours un cycle C5 ?
E = edges_of(n)
n_c5 = 0
for bits in product((0, 1), repeat=ne):
    if mono_triangle(n, bits) is None:
        red = [e for e, b in zip(E, bits) if b == 1]
        deg = [0] * n
        for a, b in red:
            deg[a] += 1
            deg[b] += 1
        if len(red) == 5 and deg == [2] * n:
            n_c5 += 1
print(f"survivantes dont le rouge est exactement un cycle C5 : {n_c5} / {cnt}")

K_5 : 10 aretes, 1024 colorations, sans triangle monochrome : 12
survivantes dont le rouge est exactement un cycle C5 : 12 / 12


### Lecture du résultat

Douze survivantes sur $1024$ : $R(3,3) > 5$. Et la structure est totale — la mesure le
dit : **chacune des 12 survivantes a pour rouge un cycle $C_5$ exactement** (cinq
arêtes, chaque sommet de degré 2), le bleu étant le $C_5$ complémentaire. Ce n'est pas
un hasard d'échantillon : 12 est précisément le nombre de cycles hamiltoniens étiquetés
de $K_5$, soit $\frac{5!}{2 \cdot 5} = 12$. Survivre à Ramsey sur $K_5$, c'est
**être** un double pentagone — la borne constructive de ce côté est un objet unique,
douze fois déguisé.

In [3]:
n = 6
cnt, _ = count_avoiders(n)
ne = len(edges_of(n))
print(f"K_{n} : {ne} aretes, {2**ne} colorations, sans triangle monochrome : {cnt}")

# Ce que devient une coloration quelconque : ses triangles monochromes
bits = (0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0)  # 15 bits arbitraires
E = edges_of(n)
tris = []
for tri in combinations(range(n), 3):
    cols = [bits[E.index((min(a, b), max(a, b)))] for a, b in combinations(tri, 2)]
    if cols[0] == cols[1] == cols[2]:
        tris.append(('rouge' if cols[0] else 'bleu', tri))
print(f"\ncoloration arbitraire {bits} : {len(tris)} triangles monochromes")
for couleur, tri in tris[:4]:
    print(f"  {couleur} : {tri}")

K_6 : 15 aretes, 32768 colorations, sans triangle monochrome : 0

coloration arbitraire (0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0) : 4 triangles monochromes
  bleu : (0, 1, 5)
  rouge : (0, 2, 3)
  bleu : (0, 4, 5)
  rouge : (1, 2, 4)


### Verdict — $R(3,3) = 6$

Zéro survivante sur $32768$ : **toute** coloration de $K_6$ porte un triangle
monochrome, et $K_5$ en évitait encore douze. L'énumération intégrale est une preuve
complète : $R(3,3) = 6$, exactement. Notons l'asymétrie du coût — $K_5$ a demandé
$1024$ vérifications, $K_6$ en demande $32768$ : un facteur 32 pour **un** sommet de
plus. Chaque palier multiplie le monde par $2^{n}$ environ. C'est la première fois que
le mur se montre, et il n'est pas encore haut.

## 2. Le pont comptage → existence : l'argument d'Erdős

Pour $R(4,4)$, la même preuve demanderait d'épuiser les colorations de $K_{18}$ (la
valeur exacte $R(4,4) = 18$ est connue — mais pas par épuisement naïf : la cellule
ci-dessous chiffre l'obstacle). L'argument probabiliste d'Erdős (1947) renverse la
question : pour montrer qu'une coloration **sans** $K_k$ monochrome existe, nul besoin
de la trouver — il suffit de montrer que les colorations qui en contiennent un sont
**moins nombreuses que le total**.

La forme générale : si $\binom{n}{k} \cdot 2^{1 - \binom{k}{2}} < 1$, alors
$R(k,k) > n$. La quantité de gauche majore le nombre **attendu** de copies
monochromes de $K_k$ dans une coloration aléatoire de $K_n$ ; si elle est inférieure
à 1, une coloration sans aucune copie existe forcément.

In [4]:
import math

# Encadrements connus : survey dynamique DS1 de Radziszowski (Electronic J. Combinatorics)
known = {3: "6 (exact)", 4: "18 (exact)", 5: "[43, 48]",
         6: "[102, 165]", 7: "[205, 540]", 8: "[281, 1870]"}

print(f"{'k':>2} {'Erdos donne':>13} {'R(k,k) connu':>13} {'mur 2^C(k,2)':>12}")
for k in range(3, 9):
    n = 3
    while math.comb(n, k) * 2 ** (1 - math.comb(k, 2)) < 1:
        n += 1
    print(f"{k:>2} {'R > ' + str(n - 1):>13} {known[k]:>13} {2.0 ** math.comb(k, 2):>12.2e}")

 k   Erdos donne  R(k,k) connu mur 2^C(k,2)
 3         R > 3     6 (exact)     8.00e+00
 4         R > 6    18 (exact)     6.40e+01
 5        R > 11      [43, 48]     1.02e+03
 6        R > 17    [102, 165]     3.28e+04
 7        R > 27    [205, 540]     2.10e+06
 8        R > 42   [281, 1870]     2.68e+08


### Lecture du résultat

La borne donne $R(3,3) > 3$ — médiocre : l'épuisement de la section 1 a donné $> 5$.
Mais regardez la colonne du mur : le monde à énumérer double à chaque palier de $k$
croissant, et l'encadrement connu s'élargit en proportion ($[43, 48]$ pour $k = 5$ :
cinq valeurs possibles, personne ne sait trancher). La borne probabiliste est
**faible en petite dimension** — elle ne remplace pas l'énumération là où
l'énumération est possible — et c'est exactement pourquoi les deux coexistent : le
comptage exact règne jusqu'où il atteint, l'argument de comptage **majoré** règne
au-delà.

In [5]:
# Le mur de R(4,4) : que couterait l'enumeration naive de K_18 ?
colorings = 2.0 ** 153
rate = 1e9                      # 1 milliard de colorations par seconde (optimiste)
seconds = colorings / rate
years = seconds / (365.25 * 24 * 3600)
age_universe = 1.38e10          # annees
print(f"colorations de K_18 : 2^153 = {colorings:.3e}")
print(f"a 1e9 colorations/s : {years:.2e} annees")
print(f"age de l'univers    : {age_universe:.2e} annees")
print(f"rapport             : {years / age_universe:.1e} x l'age de l'univers")

colorations de K_18 : 2^153 = 1.142e+46
a 1e9 colorations/s : 3.62e+29 annees
age de l'univers    : 1.38e+10 annees
rapport             : 2.6e+19 x l'age de l'univers


### Verdict — le pont comptage → existence

Même à un milliard de colorations par seconde, épuiser $K_{18}$ demande de l'ordre de
$10^{19}$ âges de l'univers (rapport exact imprimé ci-dessus) : le geste qui a prouvé
$R(3,3) = 6$ est **mort** à partir de là. L'inégalité d'Erdős — une ligne — donne
quant à elle $R(k,k) > n$ pour tout $n$ vérifiant la condition : c'est le pont. Ce que
l'énumération obtenait par constat exhaustif (« zéro survivante »), la probabilistique
l'obtient par majoration. La valeur exacte $R(4,4) = 18$ a été établie en 1955 par
Greenwood et Gleason — par une preuve combinatoire, pas par force brute.

## 3. Van der Waerden $W(2,3)$ : le même geste, un autre objet

**Énoncé.** Colorons chaque entier de $[1, n]$ en rouge ou bleu. $W(2,3)$ est le plus
petit $n$ tel que **toute** coloration contienne une progression arithmétique
monochrome de longueur 3 — trois entiers $a, a+d, a+2d$ de la même couleur. Le
comptage est le même qu'en section 1 ($2^n$ colorations à vérifier une à une) ; seul
le motif cherché change : la structure passe du graphe à la droite arithmétique.

In [6]:
def mono_ap(m, bits):
    # premiere progression arithmetique monochrome de longueur 3 (None sinon)
    for a in range(m):
        for d in range(1, m):
            if a + 2 * d >= m:
                break
            if bits[a] == bits[a + d] == bits[a + 2 * d]:
                return (a, a + d, a + 2 * d)
    return None

m = 8
avoiders = [''.join(map(str, bits))
            for bits in product((0, 1), repeat=m) if mono_ap(m, bits) is None]
print(f"[1..{m}] : {2**m} colorations, sans 3-AP monochrome : {len(avoiders)}")
for w in avoiders:
    print(f"  {w}")

[1..8] : 256 colorations, sans 3-AP monochrome : 6
  00110011
  01011010
  01100110
  10011001
  10100101
  11001100


### Lecture du résultat

Six survivantes sur $256$. Et la mesure dit mieux que le compte : ce sont
**exactement** les trois motifs de période 4 — `0011`, `0101`, `0110` — répétés deux
fois, et leurs compléments. Aucune coloration « irrégulière » ne survit : la seule
liberté laissée par la contrainte sur $[1..8]$ est un choix de motif, une phase et une
inversion des couleurs. C'est l'analogue exact du $C_5$ de la section 1 : la contrainte
ne laisse passer qu'une **forme**, en petit nombre d'exemplaires.

In [7]:
# La chute : nombre d'evitantes pour chaque taille m = 1..9
for m in range(1, 10):
    cnt = sum(1 for bits in product((0, 1), repeat=m) if mono_ap(m, bits) is None)
    print(f"m = {m} : {2**m:>4} colorations -> {cnt:>3} evitantes")

m = 1 :    2 colorations ->   2 evitantes
m = 2 :    4 colorations ->   4 evitantes
m = 3 :    8 colorations ->   6 evitantes
m = 4 :   16 colorations ->  10 evitantes
m = 5 :   32 colorations ->  14 evitantes
m = 6 :   64 colorations ->  20 evitantes
m = 7 :  128 colorations ->  16 evitantes
m = 8 :  256 colorations ->   6 evitantes
m = 9 :  512 colorations ->   0 evitantes


### Lecture du résultat — la frontière exacte

La population croît d'abord (20 évitantes à $m = 6$), plafonne, puis s'effondre :
16, 6, et **zéro** à $m = 9$. Chaque entier ajouté apporte de nouvelles progressions
— donc de nouvelles contraintes — mais seulement un facteur 2 de liberté ; au-delà
d'un seuil, les contraintes gagnent. $W(2,3) = 9$ : la preuve est, comme pour
$R(3,3) = 6$, un épuisement — 512 colorations vérifiées une à une, aucune n'y
échappe. Deux mondes (graphes, arithmétique), la même méthode, deux certitudes
exactes.

## 4. Synthèse — deux certitudes exactes, un mur, un pont

In [8]:
# Recapitulatif mesure dans ce carnet
rows = [
    ("R(3,3) > 4", "K_4", "64", "18"),
    ("R(3,3) > 5", "K_5", "1024", "12 (toutes des doubles C5)"),
    ("R(3,3) = 6", "K_6", "32768", "0"),
    ("W(2,3) > 8", "[1..8]", "256", "6 (periodiques, periode 4)"),
    ("W(2,3) = 9", "[1..9]", "512", "0"),
]
print(f"{'resultat':<12} {'monde':<8} {'taille':>7}   survivantes")
for r in rows:
    print(f"{r[0]:<12} {r[1]:<8} {r[2]:>7}   {r[3]}")

resultat     monde     taille   survivantes
R(3,3) > 4   K_4           64   18
R(3,3) > 5   K_5         1024   12 (toutes des doubles C5)
R(3,3) = 6   K_6        32768   0
W(2,3) > 8   [1..8]       256   6 (periodiques, periode 4)
W(2,3) = 9   [1..9]       512   0


### Verdict de synthèse

L'énumération a livré deux théorèmes ($R(3,3) = 6$, $W(2,3) = 9$) et des bornes
constructives en témoin ($R(3,3) > 5$, $W(2,3) > 8$) — à coût dérisoire. Le même
geste sur $R(4,4)$ se heurte à $2^{153}$ : le mur. Le pont d'Erdős remplace « toutes
vérifiées » par « moins nombreuses que le tout », et l'existence repose sur une
inégalité au lieu d'un constat. C'est la leçon de structure : **le comptage exact et
la majoration probabiliste ne sont pas deux méthodes concurrentes mais deux régimes du
même geste** — l'un règne tant que le monde tient en mémoire, l'autre prend le relais
et ne rend plus la valeur exacte, seulement un encadrement ($43 \le R(5,5) \le 48$ à
ce jour, survey DS1 de Radziszowski).

## Exercices

Trois exercices, tous sur le même moteur que les sections 1 à 3. Les stubs s'exécutent
sans erreur ; à vous de remplir le calcul.

### Exercice 1 — $K_4$ : les 18 survivantes, et leurs cycles

La section 1 a compté 18 colorations de $K_4$ sans triangle monochrome. Parmi elles,
combien ont pour rouge exactement un cycle $C_4$ (quatre arêtes rouges, chaque sommet
de degré rouge 2) ? Vérifiez votre réponse contre la valeur théorique : le nombre de
cycles de longueur 4 étiquetés sur 4 sommets.

In [9]:
# Exercice 1 -- parmi les 18 evitantes de K_4, compter les rouges = C4 (a completer)
# Etape 1 : reprendre l'enumeration de count_avoiders(4) en gardant la liste des colorations.
# Etape 2 : pour chaque survivante, reconstruire les aretes rouges et les degres.
# Indice : la reponse theorique est (4-1)!/2 = 3 cycles etiquetes.
resultat = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


### Exercice 2 — Schur : le troisième monde, $S(2) = 5$

Colorons $[1, n]$ en rouge et bleu ; une solution **monochrome** de $x + y = z$ est un
trio d'une même couleur. Le nombre de Schur $S(2) = 5$ : $[1..4]$ est coloriable sans
trio monochrome, $[1..5]$ ne l'est pas. Même moteur d'énumération, troisième structure
— l'addition.

In [10]:
# Exercice 2 -- enumeration des colorations de [1..m] sans trio monochrome x+y=z (a completer)
# Etape 1 : lister les solutions (x, y, z) avec x + y = z dans [1..m] (attention : 1+2 et 2+1
#            sont deux solutions distinctes, mais une seule contrainte).
# Etape 2 : enumerer les 2^m colorations, garder celles sans trio monochrome.
# Indice : il en reste 2 sur [1..4], et 0 sur [1..5].
def schur_avoiders(m):
    # TODO etudiant : renvoyer la liste des colorations (tuples de bits) sans trio mono
    return None

print("Exercice a completer")

Exercice a completer


### Exercice 3 — Jusqu'où l'inégalité d'Erdős grimpe-t-elle ?

La section 2 s'arrête à $k = 8$. Prolongez la table : $k = 9$ à $12$, borne d'Erdős
d'un côté ; de l'autre, l'encadrement courant de $R(k,k)$ tel que publié dans la
survey dynamique DS1 de Radziszowski (*Small Ramsey Numbers*, Electronic Journal of
Combinatorics — allez le chercher : c'est l'exercice de lecture). Que fait le
**rapport** entre la borne d'Erdős et la meilleure borne inférieure connue quand $k$
croît ?

In [11]:
# Exercice 3 -- table d'Erdos pour k = 9..12 (a completer)
# Etape 1 : reprendre la boucle de la section 2 avec range(9, 13).
# Etape 2 : ajouter log2 de la borne -- la croissance est-elle lineaire en k ?
# Indice : la borne croit comme 2^{k/2} environ ; comparez au mur 2^{C(k,2)}.
bornes = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


## Conclusion

Ce carnet a prouvé deux théorèmes en comptant tout, mesuré le mur où ce geste meurt
($2^{153}$ colorations pour $K_{18}$), et traversé ce mur par l'inégalité d'Erdős.
Trois structures — triangle de graphe, progression arithmétique, équation additive —
un seul moteur : énumérer, compter, conclure.

Références :

- **Erdős, P. (1947).** *Some remarks on the theory of graphs.* Bulletin of the
  American Mathematical Society. — l'argument probabiliste fondateur.
- **Greenwood, R. E. & Gleason, A. M. (1955).** *Combinatorial relations and
  chromatic graphs.* Canadian Journal of Mathematics. — $R(3,3) = 6$, $R(4,4) = 18$
  établis combinatoirement.
- **Radziszowski, S.** *Small Ramsey Numbers.* Electronic Journal of Combinatorics,
  Dynamic Survey DS1 (révisions successives). — l'état courant des encadrements.
- **van der Waerden, B. L. (1927).** *Beweis einer Baudetschen Vermutung.* Nieuw
  Archief voor Wiskunde. — le théorème, dans sa formulation d'origine.